In [ ]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = os.getcwd() if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

In [ ]:
!pip install optuna
import optuna

In [ ]:
import importlib
import scipy.sparse as sps
import pandas as pd
import numpy as np

from Challenge import paths
importlib.reload(paths)

from Challenge.hyper_tuning import hyperparameter_tuning

In [ ]:
# Load datasets
URM_train = sps.load_npz(paths.URM_TRAIN)
URM_validation = sps.load_npz(paths.URM_VALIDATION)

In [ ]:
def evaluate_recommender(recommender, at):
    cumulative_recall = 0.0
    num_eval = 0
    
    for user_id in range(URM_validation.shape[0]):
        relevant_items = URM_validation.indices[URM_validation.indptr[user_id]:URM_validation.indptr[user_id+1]]
        
        if len(relevant_items)>0:
            num_eval+=1
            
            recommended_items = recommender.recommend(user_id, cutoff=at)
            
            is_relevant = np.isin(recommended_items, relevant_items, assume_unique=True)
            recall_score = np.sum(is_relevant, dtype=np.float32) / relevant_items.shape[0]

            cumulative_recall += recall_score

    return cumulative_recall / num_eval

In [ ]:
# Define objective function for hyperparameter tuning
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender

STUDY_NAME = UserKNNCFRecommender.RECOMMENDER_NAME + "_hyperparameter_tuning"

def objective_function(optuna_trial: optuna.trial.Trial) -> float:
    recommender_instance = UserKNNCFRecommender(URM_train)
    recommender_instance.fit(
        topK=optuna_trial.suggest_int("topK", 10, 500),
        shrink=optuna_trial.suggest_int("shrink", 0, 1000),
        similarity=optuna_trial.suggest_categorical("similarity", ["cosine", "pearson", "jaccard", "tanimoto", "adjusted", "euclidean"]),
        normalize=optuna_trial.suggest_categorical("normalize", [True, False]),
        feature_weighting=optuna_trial.suggest_categorical("feature_weighting", ["BM25", "TF-IDF", "none"])
    )

    return evaluate_recommender(recommender_instance, at=20)

In [ ]:
# Perform hyperparameter tuning
save_results, optuna_study = hyperparameter_tuning(
    objective_function,
    study_name=STUDY_NAME,
    n_trials=100
)

In [ ]:
optuna.visualization.plot_optimization_history(optuna_study)

In [ ]:
optuna.visualization.plot_param_importances(optuna_study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(optuna_study)

In [ ]:
# Train final model on train + validation with best hyperparameters
recommender = UserKNNCFRecommender(URM_train + URM_validation)
recommender.fit(
    topK=optuna_study.best_trial.params["topK"],
    shrink=optuna_study.best_trial.params["shrink"],
    similarity=optuna_study.best_trial.params["similarity"],
    normalize=optuna_study.best_trial.params["normalize"],
    feature_weighting=optuna_study.best_trial.params["feature_weighting"]
)

# Save the trained model
recommender.save_model(paths.MODEL_DIR)

In [ ]:
submission_name = "UserKNN.csv"

# Generate recommendations for the test set
user_ids_test = pd.read_csv(paths.CHALLENGE_USER_IDS_TEST)
ids = user_ids_test["user_id"].values

recommendations = recommender.recommend(ids, cutoff=20)

os.makedirs(paths.SUBMISSIONS, exist_ok=True)
with open(os.path.join(paths.SUBMISSIONS, submission_name), "w") as f:
    f.write("user_id,item_list\n")
    for user_id, rec_list in zip(ids, recommendations):
        f.write(f"{user_id},{' '.join([str(item) for item in rec_list])}\n")